In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from tqdm import tqdm

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [3]:
def load_texts(folder, label):
    texts, labels = [], []
    for file in os.listdir(folder):
        if file.endswith(".txt"):
            with open(os.path.join(folder, file), encoding="utf-8") as f:
                text = f.read()
                text = text.split('\n', 1)[-1]
                texts.append(text)
                labels.append(label)
    return texts, labels

In [4]:
!apt-get install unrar
!unrar x origs_NER.rar
!unrar x trans_NER.rar

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from origs_NER.rar

Creating    content                                                   OK
Creating    content/origs_NER                                         OK
Extracting  content/origs_NER/130403.txt                                   0%  OK 
Extracting  content/origs_NER/81070.txt                                    0%  OK 
Extracting  content/origs_NER/275073.txt                                   0%  OK 
Extracting  content/origs_NER/266250.txt                                   0%  OK 
Extracting  content/origs_NER/67789.txt                                    0%  OK 
Extracting  content/origs_NER/49812.txt                             

In [5]:
orig_texts,  orig_labels  = load_texts("/content/content/origs_NER", 0)
trans_texts, trans_labels = load_texts("/content/content/trans_NER", 1)

In [6]:
texts = orig_texts + trans_texts
labels = orig_labels + trans_labels

In [7]:
df = pd.DataFrame({"text": texts, "label": labels})

In [8]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"], df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=SEED
)

In [9]:
MAX_LENGTH = 512
STRIDE = 256

In [10]:
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [11]:
def tokenize_three_chunks(texts, labels):
    """
    Берёт три фрагмента по 512 токенов: начало, середину и конец.
    Если текст короткий — только один фрагмент.
    Намного быстрее sliding window, при этом покрывает весь текст.
    """
    all_encodings, all_labels, all_text_ids = [], [], []
    window_size = MAX_LENGTH - 2

    for text_id, (text, label) in enumerate(tqdm(
        zip(texts, labels), total=len(texts), desc="Tokenizing"
    )):
        tokens = tokenizer.encode(
            text, add_special_tokens=False, truncation=False, verbose=False
        )

        # Определяем позиции фрагментов
        positions = [0]  # начало
        if len(tokens) > window_size:
            positions.append((len(tokens) - window_size) // 2)  # середина
        if len(tokens) > window_size * 2:
            positions.append(len(tokens) - window_size)          # конец

        for start in positions:
            chunk = tokens[start : start + window_size]
            input_ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
            all_encodings.append({
                "input_ids":      input_ids,
                "attention_mask": [1] * len(input_ids),
            })
            all_labels.append(label)
            all_text_ids.append(text_id)

    return all_encodings, all_labels, all_text_ids

In [12]:
train_encodings, train_labels_sw, train_text_ids = tokenize_three_chunks(
    train_texts.tolist(), train_labels.tolist()
)

Tokenizing: 100%|██████████| 1368/1368 [04:49<00:00,  4.72it/s]


In [13]:
test_encodings, test_labels_sw, test_text_ids = tokenize_three_chunks(
    test_texts.tolist(), test_labels.tolist()
)

Tokenizing: 100%|██████████| 342/342 [01:20<00:00,  4.27it/s]


In [14]:
print(f"Train фрагментов: {len(train_encodings)} (из {len(train_texts)} текстов)")
print(f"Test фрагментов:  {len(test_encodings)} (из {len(test_texts)} текстов)")

Train фрагментов: 4028 (из 1368 текстов)
Test фрагментов:  1010 (из 342 текстов)


In [36]:
class FanficDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(self.encodings[idx][key]) for key in self.encodings[idx]}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [16]:
train_dataset = FanficDataset(train_encodings, train_labels_sw)
test_dataset  = FanficDataset(test_encodings,  test_labels_sw)

In [17]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [18]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy":   accuracy_score(labels, preds),
        "f1_binary":  f1_score(labels, preds, average="binary"),  # класс 1 = перевод
        "f1_macro":   f1_score(labels, preds, average="macro"),    # среднее по классам
    }

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    "cointegrated/rubert-tiny",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/47.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider train

In [21]:
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [22]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
)

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [23]:
print(f"Фрагментов в train: {len(train_dataset)}")
print(f"Шагов на эпоху: {len(train_dataset) // 8}")
print(f"Всего шагов: {len(train_dataset) // 8 * 3}")

Фрагментов в train: 4028
Шагов на эпоху: 503
Всего шагов: 1509


In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Binary,F1 Macro
1,0.612206,0.496204,0.753465,0.731392,0.751789
2,0.461525,0.447323,0.803960,0.796296,0.803682
3,0.407371,0.436782,0.806931,0.802831,0.806847


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

TrainOutput(global_step=1512, training_loss=0.49334743350901933, metrics={'train_runtime': 100.9808, 'train_samples_per_second': 119.666, 'train_steps_per_second': 14.973, 'total_flos': 89109917097984.0, 'train_loss': 0.49334743350901933, 'epoch': 3.0})

In [28]:
print("\nПредсказание с агрегацией по текстам...")

raw = trainer.predict(test_dataset)
logits    = raw.predictions
frag_true = raw.label_ids


n_texts = len(test_texts)
text_logits = [[] for _ in range(n_texts)]
text_true   = {}

for frag_idx, text_id in enumerate(test_text_ids):
    text_logits[text_id].append(logits[frag_idx])
    text_true[text_id] = frag_true[frag_idx]


final_preds = []
final_true  = []

for text_id in range(n_texts):
    if not text_logits[text_id]:
        continue
    avg_logit = np.mean(text_logits[text_id], axis=0)
    final_preds.append(int(np.argmax(avg_logit)))
    final_true.append(text_true[text_id])

print(f"\nРезультаты на уровне текстов (n={len(final_true)}):")
print(f"  Accuracy  : {accuracy_score(final_true, final_preds):.4f}")
print(f"  F1 binary : {f1_score(final_true, final_preds, average='binary'):.4f}")
print(f"  F1 macro  : {f1_score(final_true, final_preds, average='macro'):.4f}")


Предсказание с агрегацией по текстам...



Результаты на уровне текстов (n=342):
  Accuracy  : 0.8713
  F1 binary : 0.8683
  F1 macro  : 0.8713


In [29]:
def tokenize_with_sliding_window(texts, labels):
    all_encodings = []
    all_labels = []
    all_text_ids = []

    window_size = MAX_LENGTH - 2

    for text_id, (text, label) in enumerate(tqdm(
        zip(texts, labels), total=len(texts), desc="Tokenizing"
    )):

        tokens = tokenizer.encode(
            text,
            add_special_tokens=False,
            truncation=False,
            verbose=False,
        )

        start = 0
        while start < max(len(tokens), 1):
            chunk = tokens[start : start + window_size]
            input_ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
            all_encodings.append({
                "input_ids": input_ids,
                "attention_mask": [1] * len(input_ids),
            })
            all_labels.append(label)
            all_text_ids.append(text_id)

            if start + window_size >= len(tokens):
                break
            start += STRIDE

    return all_encodings, all_labels, all_text_ids

In [30]:
train_encodings, train_labels_sw, train_text_ids = tokenize_with_sliding_window(
    train_texts.tolist(), train_labels.tolist()
)

Tokenizing: 100%|██████████| 1368/1368 [05:16<00:00,  4.32it/s]


In [34]:
test_encodings, test_labels_sw, test_text_ids = tokenize_with_sliding_window(
    test_texts.tolist(), test_labels.tolist()
)

Tokenizing: 100%|██████████| 342/342 [01:41<00:00,  3.37it/s]


In [35]:
print(f"Train фрагментов: {len(train_encodings)} (из {len(train_texts)} текстов)")
print(f"Test фрагментов:  {len(test_encodings)} (из {len(test_texts)} текстов)")

Train фрагментов: 370833 (из 1368 текстов)
Test фрагментов:  107795 (из 342 текстов)


In [37]:
train_dataset = FanficDataset(train_encodings, train_labels_sw)
test_dataset  = FanficDataset(test_encodings,  test_labels_sw)

In [38]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [39]:
training_args = TrainingArguments(
    output_dir="./results1",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
)

In [40]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [41]:
print(f"Фрагментов в train: {len(train_dataset)}")
print(f"Шагов на эпоху: {len(train_dataset) // 8}")
print(f"Всего шагов: {len(train_dataset) // 8 * 3}")

Фрагментов в train: 370833
Шагов на эпоху: 46354
Всего шагов: 139062


In [42]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Binary,F1 Macro
1,0.278991,0.341800,0.898335,0.905075,0.897820
2,0.260038,0.348763,0.909430,0.918966,0.908158
3,0.214095,0.392720,0.910116,0.920557,0.908537


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

TrainOutput(global_step=139065, training_loss=0.2636381482236213, metrics={'train_runtime': 9269.0445, 'train_samples_per_second': 120.023, 'train_steps_per_second': 15.003, 'total_flos': 8203797886593024.0, 'train_loss': 0.2636381482236213, 'epoch': 3.0})

In [43]:
print("\nПредсказание с агрегацией по текстам...")

raw = trainer.predict(test_dataset)
logits    = raw.predictions
frag_true = raw.label_ids

n_texts = len(test_texts)
text_logits = [[] for _ in range(n_texts)]
text_true   = {}

for frag_idx, text_id in enumerate(test_text_ids):
    text_logits[text_id].append(logits[frag_idx])
    text_true[text_id] = frag_true[frag_idx]

final_preds = []
final_true  = []

for text_id in range(n_texts):
    if not text_logits[text_id]:
        continue
    avg_logit = np.mean(text_logits[text_id], axis=0)
    final_preds.append(int(np.argmax(avg_logit)))
    final_true.append(text_true[text_id])

print(f"\nРезультаты на уровне текстов (n={len(final_true)}):")
print(f"  Accuracy  : {accuracy_score(final_true, final_preds):.4f}")
print(f"  F1 binary : {f1_score(final_true, final_preds, average='binary'):.4f}")
print(f"  F1 macro  : {f1_score(final_true, final_preds, average='macro'):.4f}")


Предсказание с агрегацией по текстам...



Результаты на уровне текстов (n=342):
  Accuracy  : 0.9649
  F1 binary : 0.9647
  F1 macro  : 0.9649
